In [ ]:
# If needed, uncomment:
# !pip install -q pandas numpy matplotlib seaborn

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

# ================================================================
# 1. LOAD DATA
# ================================================================
file_path = "Table1.csv"

try:
    df_raw = pd.read_csv(file_path, low_memory=False)
    print(f"File loaded successfully: {file_path}")
    print(f"Rows: {df_raw.shape[0]} | Columns: {df_raw.shape[1]}")
except FileNotFoundError:
    print(f"Error: file {file_path} was not found.")
    raise

# ================================================================
# 2. TRANSLATION MAPS
# ================================================================
specialty_translation = {
    "Psiquiatria": "Psychiatry",
    "Psicologia": "Psychology",
    "Saúde mental": "Mental Health",
    "Oftalmologia": "Ophthalmology",
    "Ortopedia": "Orthopedics",
    "Endocrinologia": "Endocrinology",
    "Infectologia": "Infectious Diseases",
    "Cirurgia geral": "General Surgery",
    "Cardiologia": "Cardiology",
    "Dermatologia": "Dermatology",
    "Neurologia": "Neurology",
    "Urologia": "Urology",
    "Ginecologia": "Gynecology",
    "Pneumologia": "Pulmonology",
}

sex_translation = {
    "Masculino": "Male",
    "Feminino": "Female",
    "Homem": "Male",
    "Mulher": "Female",
    "M": "Male",
    "F": "Female",
}

substance_translation = {
    "Álcool": "Alcohol",
    "Tabaco": "Tobacco",
    "Maconha": "Cannabis",
    "Drogas Pesadas (Crack/Cocaína/Opioides)": "Heavy Drugs (Crack/Cocaine/Opioids)",
}

ciap_translation = {
    "K86-HIPERTENSÃO SEM COMPLICAÇÕES": "K86-HYPERTENSION WITHOUT COMPLICATIONS",
    "P01-SENSAÇÃO DE ANSIEDADE/NERVOSISMO/TENSÃO": "P01-FEELING ANXIOUS/NERVOUS/TENSE",
    "P06-PERTURBAÇÃO DO SONO": "P06-SLEEP DISTURBANCE",
    "R05-TOSSE": "R05-COUGH",
    "N01-CEFALÉIA": "N01-HEADACHE",
    "P19-ABUSO DE DROGAS": "P19-DRUG ABUSE",
    "L18-DORES MUSCULARES": "L18-MUSCLE PAIN",
    "P76-DEPRESSÃO": "P76-DEPRESSION",
    "R02-FALTA DE AR/DISPNEIA": "R02-SHORTNESS OF BREATH/DYSPNEA",
    "A04-FRAQUEZA/CANSAÇO GERAL": "A04-GENERAL WEAKNESS/FATIGUE",
    "A03-FEBRE": "A03-FEVER",
    "D01-DOR ABDOMINAL/CÓLICA": "D01-ABDOMINAL PAIN/CRAMPS",
    "L03-SINTOMAS/QUEIXAS LOMBARES": "L03-LOW BACK SYMPTOMS/COMPLAINTS",
    "R74-INFECÇÃO AGUDA DO TRATO RESPIRATÓRIO SUPERIOR": "R74-ACUTE UPPER RESPIRATORY TRACT INFECTION",
    "T90-DIABETES NÃO INSULINO-DEPENDENTE": "T90-NON-INSULIN-DEPENDENT DIABETES",
    "P99-OUTRAS PERTURBAÇÕES PSICOLÓGICAS": "P99-OTHER PSYCHOLOGICAL DISORDERS",
    "P98-OUTRAS PSICOSES NE": "P98-OTHER PSYCHOSES, NOT ELSEWHERE CLASSIFIED",
    "P74-DISTÚRBIO ANSIOSO/ESTADO DE ANSIEDADE": "P74-ANXIETY DISORDER / ANXIETY STATE",
    "P72-ESQUIZOFRENIA": "P72-SCHIZOPHRENIA",

    "F05-OUTRAS PERTURBAÇÕES VISUAIS": "F05-OTHER VISUAL DISTURBANCES",
    "F99-OUTRA DOENÇAS OCULARES/ANEXOS": "F99-OTHER EYE/ADNEXAL DISEASES",
    "F91-ERRO DE REFRAÇÃO": "F91-REFRACTIVE ERROR",

    "L76-OUTRAS FRATURAS": "L76-OTHER FRACTURES",
    "L14-SINAIS/SINTOMAS DA COXA/PERNA": "L14-THIGH/LEG SIGNS AND SYMPTOMS",
    "L73-FRATURA: TÍBIA/PERÔNIO/ FÍBULA": "L73-FRACTURE: TIBIA/FIBULA",
    "L09-SINAIS/SINTOMAS DOS BRAÇOS": "L09-ARM SIGNS AND SYMPTOMS",

    "D89-HÉRNIA INGUINAL": "D89-INGUINAL HERNIA",
    "D91-HÉRNIA ABDOMINAL, OUTRAS": "D91-OTHER ABDOMINAL HERNIA",
    "S85-CISTO PILONIDAL/FISTULA": "S85-PILONIDAL CYST/FISTULA",
    "S10-FURÚNCULO/CARBÚNCULO": "S10-FURUNCLE/CARBUNCLE",
}

def translate_specialty(name: str) -> str:
    name = str(name).strip()
    return specialty_translation.get(name, name)

def translate_sex(value: str) -> str:
    value = str(value).strip()
    return sex_translation.get(value, value)

def translate_ciap(value: str) -> str:
    value = str(value).strip()
    return ciap_translation.get(value, value)

def binarize_checked(series: pd.Series) -> pd.Series:
    return series.astype(str).str.lower().str.strip().isin(
        ["checked", "1", "1.0", "sim", "verdadeiro", "true", "yes"]
    ).astype(int)

# ================================================================
# 3. SHARED HELPERS
# ================================================================
def get_specialty_columns(df: pd.DataFrame):
    return [c for c in df.columns if ("encaminhamento" in c.lower() or "especialidade" in c.lower()) and "choice=" in c.lower()]

def get_ciap_column(df: pd.DataFrame):
    cols = [c for c in df.columns if "ciap" in c.lower()]
    return cols[0] if cols else None

def get_age_column(df: pd.DataFrame):
    cols = [c for c in df.columns if "idade" in c.lower()]
    return cols[0] if cols else None

def get_sex_column(df: pd.DataFrame):
    cols = [c for c in df.columns if "sexo" in c.lower() or "gênero" in c.lower() or "genero" in c.lower()]
    return cols[0] if cols else None

def prepare_longitudinal_base(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df = df.sort_values(["Record ID", "Repeat Instance"], na_position="first")

    col_age = get_age_column(df)
    col_sex = get_sex_column(df)
    col_ciap = get_ciap_column(df)

    if col_age:
        df[col_age] = df.groupby("Record ID")[col_age].transform(lambda x: x.ffill().bfill())
    if col_sex:
        df[col_sex] = df.groupby("Record ID")[col_sex].transform(lambda x: x.ffill().bfill())
    if col_ciap:
        df[col_ciap] = df.groupby("Record ID")[col_ciap].transform(lambda x: x.ffill())

    return df

def build_substance_flags(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    text_cols = [c for c in df.columns if df[c].dtype == object]

    df["Use_Cannabis"] = df[text_cols].apply(
        lambda x: x.astype(str).str.contains(r"\b(maconha|cannabis)\b", case=False, na=False, regex=True)
    ).any(axis=1).astype(int)

    df["Use_Heavy_Drugs"] = df[text_cols].apply(
        lambda x: x.astype(str).str.contains(r"\b(crack|coca[íi]na|hero[íi]na|opi[óo]ide)\b", case=False, na=False, regex=True)
    ).any(axis=1).astype(int)

    alcohol_cols = [c for c in df.columns if (("álcool" in c.lower()) or ("alcool" in c.lower())) and "choice=" in c.lower()]
    tobacco_cols = [c for c in df.columns if ("tabaco" in c.lower() or "fumo" in c.lower() or "cigarro" in c.lower()) and "choice=" in c.lower()]

    df["Use_Alcohol"] = (
        df[alcohol_cols].apply(lambda col: col.astype(str).str.contains("Checked|1|sim", case=False, na=False)).any(axis=1).astype(int)
        if alcohol_cols else 0
    )
    df["Use_Tobacco"] = (
        df[tobacco_cols].apply(lambda col: col.astype(str).str.contains("Checked|1|sim", case=False, na=False)).any(axis=1).astype(int)
        if tobacco_cols else 0
    )

    return df

# ================================================================
# 4. ANALYSIS 1 — TOP SPECIALTY DEMAND: QUEUE WEIGHT VS PATIENT RISK
# ================================================================
def analysis_1_top_specialty_demand(df: pd.DataFrame) -> None:
    df = df.copy()
    cols_esp = get_specialty_columns(df)
    col_ciap = get_ciap_column(df)

    if not cols_esp:
        print("ERROR: No referral/specialty columns were found.")
        return

    for c in cols_esp:
        df[c] = binarize_checked(df[c])

    clean_names = {
        c: translate_specialty(c.split("=")[-1].replace(")", "").strip().capitalize())
        for c in cols_esp
    }

    absolute_sums = df[cols_esp].sum()
    total_referrals_generated = absolute_sums.sum()

    top_5_original = absolute_sums.sort_values(ascending=False).head(5)

    queue_weight = (top_5_original / total_referrals_generated) * 100
    queue_weight.index = [clean_names[c] for c in top_5_original.index]

    df_patients = df.groupby("Record ID")[list(top_5_original.index)].max()
    total_unique_patients = df["Record ID"].nunique()
    patients_in_top5 = df_patients.sum()

    patient_risk = (patients_in_top5 / total_unique_patients) * 100
    patient_risk.index = [clean_names[c] for c in top_5_original.index]

    demand_table = pd.DataFrame({
        "Medical Specialty": queue_weight.index,
        "Issued Referrals (N)": top_5_original.values,
        "Total Queue Weight (%)": queue_weight.values.round(2),
        "Unique Patients Referred (N)": patients_in_top5.values,
        "Population Risk (%)": patient_risk.values.round(2),
    })
    demand_table_file = "specialty_demand_base_en.csv"
    demand_table.to_csv(demand_table_file, index=False, encoding="utf-8-sig")

    audit_rows = []
    if col_ciap:
        for col_original in top_5_original.index:
            specialty_name = clean_names[col_original]
            df_filtered = df[df[col_original] == 1]
            total_cases = len(df_filtered)

            if total_cases > 0:
                top_ciaps = df_filtered[col_ciap].dropna().value_counts().head(5)
                for ciap, qty in top_ciaps.items():
                    audit_rows.append({
                        "Destination Specialty": specialty_name,
                        "Total Referrals for Specialty": total_cases,
                        "CIAP-2 Reason": translate_ciap(ciap),
                        "Absolute Frequency (N)": qty,
                        "Bottleneck Proportion (%)": round((qty / total_cases) * 100, 2),
                    })

    audit_table = pd.DataFrame(audit_rows)
    audit_table_file = "specialty_ciap_audit_base_en.csv"
    audit_table.to_csv(audit_table_file, index=False, encoding="utf-8-sig")

    print("\n" + "=" * 80)
    print("DATA TABLES EXPORTED SUCCESSFULLY:")
    print(f"- '{demand_table_file}' (queue weight and patient risk)")
    print(f"- '{audit_table_file}' (clinical bottleneck reasons)")
    print("=" * 80)

    def generate_highlight_palette(names, highlight_term="psychiatry"):
        highlight = "#e74c3c"
        neutral = "#bdc3c7"
        return [highlight if highlight_term in str(name).lower() else neutral for name in names]

    palette_a = generate_highlight_palette(queue_weight.index)
    palette_b = generate_highlight_palette(patient_risk.index)

    sns.set_theme(style="whitegrid")
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

    sns.barplot(x=queue_weight.values, y=queue_weight.index, palette=palette_a, ax=ax1)
    ax1.set_title(
        f"VIEW A: Queue Weight\n(Total: {int(total_referrals_generated)} referrals issued)",
        fontsize=14,
        fontweight="bold",
        pad=15,
    )
    ax1.set_xlabel("% Share of the Total Queue", fontsize=12, fontweight="bold")
    ax1.set_ylabel("")

    for i, p in enumerate(ax1.patches):
        text_color = "#c0392b" if palette_a[i] == "#e74c3c" else "#333333"
        ax1.annotate(
            f"{p.get_width():.1f}%",
            (p.get_width(), p.get_y() + p.get_height() / 2.0),
            ha="left",
            va="center",
            xytext=(5, 0),
            textcoords="offset points",
            fontweight="bold",
            color=text_color,
        )

    sns.barplot(x=patient_risk.values, y=patient_risk.index, palette=palette_b, ax=ax2)
    ax2.set_title(
        f"VIEW B: Patient-Level Demand\n(Base: {total_unique_patients} unique patients)",
        fontsize=14,
        fontweight="bold",
        pad=15,
    )
    ax2.set_xlabel("% of Unique Patients Who Needed the Specialty", fontsize=12, fontweight="bold")
    ax2.set_ylabel("")

    for i, p in enumerate(ax2.patches):
        text_color = "#c0392b" if palette_b[i] == "#e74c3c" else "#333333"
        ax2.annotate(
            f"{p.get_width():.1f}%",
            (p.get_width(), p.get_y() + p.get_height() / 2.0),
            ha="left",
            va="center",
            xytext=(5, 0),
            textcoords="offset points",
            fontweight="bold",
            color=text_color,
        )

    sns.despine(left=True, bottom=True)
    plt.tight_layout()
    plt.show()

    print("\n" + "=" * 90)
    print("CLINICAL AUDIT: BOTTLENECK REASONS (TOP 3 SPECIALTIES)")
    print("=" * 90)

    if col_ciap:
        for col_original in top_5_original.head(3).index:
            nice_name = clean_names[col_original]
            df_filtered = df[df[col_original] == 1]
            total_cases = len(df_filtered)
            marker = " ⭐ [HIGHLIGHT] " if "psychiatry" in nice_name.lower() else " "

            print(f"\n{marker}[ QUEUE: {nice_name.upper()} ] - {total_cases} referrals issued")
            if total_cases > 0:
                top_ciaps = df_filtered[col_ciap].dropna().value_counts().head(4)
                if len(top_ciaps) > 0:
                    for ciap, qty in top_ciaps.items():
                        pct = (qty / total_cases) * 100
                        ciap_en = translate_ciap(ciap)
                        print(f"  -> {ciap_en[:70]:<70} | {qty} cases ({pct:.1f}% of referrals)")
                else:
                    print("  -> No CIAP-2 reason recorded for these referrals.")
    print("\n" + "=" * 90 + "\n")

# ================================================================
# 5. ANALYSIS 2 — EPIDEMIOLOGICAL DOSSIER: MENTAL HEALTH QUEUE
# ================================================================
def analysis_2_mental_health_dossier(df: pd.DataFrame) -> None:
    df = prepare_longitudinal_base(df)
    col_age = get_age_column(df)
    col_sex = get_sex_column(df)
    col_ciap = get_ciap_column(df)
    cols_esp = get_specialty_columns(df)

    cols_psych = [c for c in cols_esp if "psiquiatria" in c.lower() or "saúde mental" in c.lower() or "psicologia" in c.lower()]

    if not cols_psych:
        print("ERROR: No Psychiatry/Psychology/Mental Health columns were found.")
        return

    for c in cols_psych:
        df[c] = binarize_checked(df[c])

    df["Mental_Health_Queue"] = df[cols_psych].max(axis=1)
    df_guides = df[df["Mental_Health_Queue"] == 1].copy()
    df_psych = df_guides.sort_values("Repeat Instance").groupby("Record ID").last().reset_index()

    total_patients_psych = len(df_psych)

    print("\n" + "=" * 90)
    print(f"PROCESSING DOSSIER: {total_patients_psych} unique patients found in the queue.")
    print(f"Unified columns: {[translate_specialty(c.split('=')[-1].replace(')', '').strip()) for c in cols_psych]}")
    print("=" * 90)

    if total_patients_psych == 0:
        return

    if col_age:
        df_psych["Age_Num"] = pd.to_numeric(df_psych[col_age], errors="coerce")

    cols_inf = [c for c in df.columns if "doenças infecciosas" in c.lower() and "choice=" in c.lower() and "não" not in c.lower()]
    cols_chr = [c for c in df.columns if "doenças crônicas" in c.lower() and "choice=" in c.lower() and "não" not in c.lower()]

    for c in cols_inf + cols_chr:
        df_psych[c] = binarize_checked(df_psych[c])

    df_psych["Total_Infectious"] = df_psych[cols_inf].sum(axis=1)
    df_psych["Total_Chronic"] = df_psych[cols_chr].sum(axis=1)

    df_psych = build_substance_flags(df_psych)

    if col_ciap:
        ciap_table = df_psych[col_ciap].dropna().value_counts().reset_index()
        ciap_table.columns = ["CIAP-2 Reason", "Unique Patients (N)"]
        ciap_table["CIAP-2 Reason"] = ciap_table["CIAP-2 Reason"].apply(translate_ciap)
        ciap_table["Proportion (%)"] = (ciap_table["Unique Patients (N)"] / total_patients_psych * 100).round(2)
        ciap_file = "mental_health_dossier_reasons_en.csv"
        ciap_table.to_csv(ciap_file, index=False, encoding="utf-8-sig")
    else:
        ciap_file = None

    if col_age and col_sex:
        df_demog = df_psych.dropna(subset=["Age_Num", col_sex]).copy()
        if len(df_demog) > 0:
            top_sexes = df_demog[col_sex].value_counts().head(2).index
            df_demog_filtered = df_demog[df_demog[col_sex].isin(top_sexes)].copy()
            df_demog_filtered["Sex_EN"] = df_demog_filtered[col_sex].apply(translate_sex)

            demographic_table = (
                df_demog_filtered.groupby("Sex_EN")["Age_Num"]
                .agg(["count", "mean", "median", "std", "min", "max"])
                .reset_index()
            )
            demographic_table.columns = [
                "Sex",
                "N (Patients)",
                "Mean Age",
                "Median Age",
                "Standard Deviation",
                "Minimum Age",
                "Maximum Age",
            ]
            demographic_table["Mean Age"] = demographic_table["Mean Age"].round(1)
            demographic_table["Standard Deviation"] = demographic_table["Standard Deviation"].round(2)
            demographic_file = "mental_health_dossier_demography_en.csv"
            demographic_table.to_csv(demographic_file, index=False, encoding="utf-8-sig")
        else:
            df_demog_filtered = pd.DataFrame()
            demographic_file = None
    else:
        df_demog_filtered = pd.DataFrame()
        demographic_file = None

    comorbidities = pd.DataFrame({
        "Burden Category": ["No Disease", "1 to 2 Diseases", "3+ Diseases"],
        "Infectious (N)": [
            len(df_psych[df_psych["Total_Infectious"] == 0]),
            len(df_psych[(df_psych["Total_Infectious"] >= 1) & (df_psych["Total_Infectious"] <= 2)]),
            len(df_psych[df_psych["Total_Infectious"] >= 3]),
        ],
        "Chronic (N)": [
            len(df_psych[df_psych["Total_Chronic"] == 0]),
            len(df_psych[(df_psych["Total_Chronic"] >= 1) & (df_psych["Total_Chronic"] <= 2)]),
            len(df_psych[df_psych["Total_Chronic"] >= 3]),
        ],
    })
    comorbidity_file = "mental_health_dossier_comorbidities_en.csv"
    comorbidities.to_csv(comorbidity_file, index=False, encoding="utf-8-sig")

    substance_use = {
        "Alcohol": int(df_psych["Use_Alcohol"].sum()),
        "Tobacco": int(df_psych["Use_Tobacco"].sum()),
        "Cannabis": int(df_psych["Use_Cannabis"].sum()),
        "Heavy Drugs (Crack/Cocaine/Opioids)": int(df_psych["Use_Heavy_Drugs"].sum()),
    }
    substances_table = pd.DataFrame(list(substance_use.items()), columns=["Substance", "Unique Patients (N)"])
    substances_table["Proportion (%)"] = (substances_table["Unique Patients (N)"] / total_patients_psych * 100).round(2)
    substances_table = substances_table.sort_values("Unique Patients (N)", ascending=False)
    substances_file = "mental_health_dossier_substances_en.csv"
    substances_table.to_csv(substances_file, index=False, encoding="utf-8-sig")

    print("\n" + "=" * 80)
    print("DOSSIER DATA TABLES EXPORTED SUCCESSFULLY:")
    if ciap_file:
        print(f"- '{ciap_file}'")
    if demographic_file:
        print(f"- '{demographic_file}'")
    print(f"- '{comorbidity_file}'")
    print(f"- '{substances_file}'")
    print("=" * 80)

    sns.set_theme(style="whitegrid")
    fig = plt.figure(figsize=(20, 14))
    fig.suptitle(
        f"Epidemiological Dossier: Mental Health Queue (N = {total_patients_psych} unique patients)",
        fontsize=20,
        fontweight="bold",
        y=0.98,
    )

    ax1 = plt.subplot(2, 2, 1)
    if col_ciap:
        top_ciaps = df_psych[col_ciap].dropna().value_counts().head(5)
        translated_ciaps = [translate_ciap(m) for m in top_ciaps.index]
        short_names = [str(m)[:55] + ("..." if len(str(m)) > 55 else "") for m in translated_ciaps]
        sns.barplot(x=top_ciaps.values, y=short_names, palette="Reds_r", ax=ax1)
        ax1.set_title("1. Main Referral Reasons (CIAP-2)", fontsize=14, fontweight="bold")
        ax1.set_xlabel("Number of Unique Patients")
        ax1.set_ylabel("")
        for p in ax1.patches:
            ax1.annotate(
                f" {int(p.get_width())}",
                (p.get_width(), p.get_y() + p.get_height() / 2.0),
                va="center",
                fontweight="bold",
            )

    ax2 = plt.subplot(2, 2, 2)
    if not df_demog_filtered.empty:
        sns.violinplot(data=df_demog_filtered, x="Sex_EN", y="Age_Num", palette="muted", inner="quart", ax=ax2)
        ax2.set_title("2. Demographic Profile: Age Distribution by Sex", fontsize=14, fontweight="bold")
        ax2.set_ylabel("Age (Years)")
        ax2.set_xlabel("")
    else:
        ax2.text(0.5, 0.5, "Numeric ages were not found for the patients.", ha="center", va="center")

    ax3 = plt.subplot(2, 2, 3)
    comorb_melt = comorbidities.melt(id_vars="Burden Category", var_name="Type", value_name="Patients")
    sns.barplot(data=comorb_melt, x="Burden Category", y="Patients", hue="Type", palette=["#e67e22", "#2980b9"], ax=ax3)
    ax3.set_title("3. Physical Disease Burden (Comorbidities)", fontsize=14, fontweight="bold")
    ax3.set_ylabel("Number of Patients")
    ax3.set_xlabel("")

    ax4 = plt.subplot(2, 2, 4)
    sr_substances = pd.Series(substance_use).sort_values(ascending=False)
    sns.barplot(x=sr_substances.values, y=sr_substances.index, palette="viridis", ax=ax4)
    ax4.set_title("4. Chemical Map: Substance Use in the Queue", fontsize=14, fontweight="bold")
    ax4.set_xlabel("Number of Unique Patients")
    ax4.set_ylabel("")
    for p in ax4.patches:
        ax4.annotate(
            f" {int(p.get_width())}",
            (p.get_width(), p.get_y() + p.get_height() / 2.0),
            va="center",
            fontweight="bold",
        )

    sns.despine(left=True, bottom=True)
    plt.tight_layout(pad=3.0)
    plt.show()

    print("\n[ REASON CHECK (CIAP-2) ]")
    if col_ciap:
        top_ciaps_console = df_psych[col_ciap].dropna().value_counts().head(8)
        for ciap, qty in top_ciaps_console.items():
            pct = (qty / total_patients_psych) * 100
            print(f"  -> {translate_ciap(ciap)[:75]:<75} | {qty} patients ({pct:.1f}%)")

    print("\n[ AGE CHECK ]")
    if not df_demog_filtered.empty:
        print(f"  -> Mean age calculated: {df_demog_filtered['Age_Num'].mean():.1f} years")
        print(f"  -> Youngest patient: {df_demog_filtered['Age_Num'].min():.0f} years")
        print(f"  -> Oldest patient: {df_demog_filtered['Age_Num'].max():.0f} years")
    print("=" * 90 + "\n")

# ================================================================
# 6. ANALYSIS 3 — EPIDEMIOLOGICAL DOSSIERS: OPHTHALMOLOGY AND ORTHOPEDICS
# ================================================================
def analysis_3_target_specialty_dossiers(df: pd.DataFrame) -> None:
    df = prepare_longitudinal_base(df)
    col_age = get_age_column(df)
    col_sex = get_sex_column(df)
    col_ciap = get_ciap_column(df)
    cols_esp = get_specialty_columns(df)

    target_specialties = ["oftalmologia", "ortopedia"]

    for target in target_specialties:
        col_target = next((c for c in cols_esp if target in c.lower()), None)

        if not col_target:
            print(f"\n[ERROR] Column for specialty '{target.upper()}' was not found in the dataset.")
            continue

        df_local = df.copy()
        df_local["Current_Queue"] = binarize_checked(df_local[col_target])

        df_guides = df_local[df_local["Current_Queue"] == 1].copy()
        df_target = df_guides.sort_values("Repeat Instance").groupby("Record ID").last().reset_index()

        total_patients = len(df_target)
        specialty_name = translate_specialty(target.capitalize())

        print("\n" + "=" * 100)
        print(f"GENERATING DOSSIER AND TABLES: {specialty_name.upper()} | {total_patients} unique patients in queue.")
        print("=" * 100)

        if total_patients == 0:
            continue

        if col_age:
            df_target["Age_Num"] = pd.to_numeric(df_target[col_age], errors="coerce")

        cols_inf = [c for c in df_local.columns if "doenças infecciosas" in c.lower() and "choice=" in c.lower() and "não" not in c.lower()]
        cols_chr = [c for c in df_local.columns if "doenças crônicas" in c.lower() and "choice=" in c.lower() and "não" not in c.lower()]

        for c in cols_inf + cols_chr:
            df_target[c] = binarize_checked(df_target[c])

        df_target["Total_Infectious"] = df_target[cols_inf].sum(axis=1)
        df_target["Total_Chronic"] = df_target[cols_chr].sum(axis=1)

        df_target = build_substance_flags(df_target)

        comorbidities = pd.DataFrame({
            "Burden Category": ["No Physical Disease", "1 to 2 Diseases", "3+ Diseases"],
            "Infectious (N)": [
                len(df_target[df_target["Total_Infectious"] == 0]),
                len(df_target[(df_target["Total_Infectious"] >= 1) & (df_target["Total_Infectious"] <= 2)]),
                len(df_target[df_target["Total_Infectious"] >= 3]),
            ],
            "Chronic (N)": [
                len(df_target[df_target["Total_Chronic"] == 0]),
                len(df_target[(df_target["Total_Chronic"] >= 1) & (df_target["Total_Chronic"] <= 2)]),
                len(df_target[df_target["Total_Chronic"] >= 3]),
            ],
        })

        substance_use = {
            "Alcohol": int(df_target["Use_Alcohol"].sum()),
            "Tobacco": int(df_target["Use_Tobacco"].sum()),
            "Cannabis": int(df_target["Use_Cannabis"].sum()),
            "Heavy Drugs (Crack/Cocaine/Opioids)": int(df_target["Use_Heavy_Drugs"].sum()),
        }

        if col_ciap:
            ciap_table = df_target[col_ciap].dropna().value_counts().reset_index()
            ciap_table.columns = ["CIAP-2 Reason", "Unique Patients (N)"]
            ciap_table["CIAP-2 Reason"] = ciap_table["CIAP-2 Reason"].apply(translate_ciap)
            ciap_table["Proportion (%)"] = (ciap_table["Unique Patients (N)"] / total_patients * 100).round(2)
            ciap_file = f"dossier_reasons_{target}_en.csv"
            ciap_table.to_csv(ciap_file, index=False, encoding="utf-8-sig")
        else:
            ciap_file = None

        if col_age and col_sex:
            df_demog_export = df_target.dropna(subset=["Age_Num", col_sex]).copy()
            if len(df_demog_export) > 0:
                top_sexes_export = df_demog_export[col_sex].value_counts().head(2).index
                df_demog_filtered = df_demog_export[df_demog_export[col_sex].isin(top_sexes_export)].copy()
                df_demog_filtered["Sex_EN"] = df_demog_filtered[col_sex].apply(translate_sex)

                demographic_table = (
                    df_demog_filtered.groupby("Sex_EN")["Age_Num"]
                    .agg(["count", "mean", "median", "std", "min", "max"])
                    .reset_index()
                )
                demographic_table.columns = [
                    "Sex", "N (Patients)", "Mean Age", "Median Age",
                    "Standard Deviation", "Minimum Age", "Maximum Age"
                ]
                demographic_table["Mean Age"] = demographic_table["Mean Age"].round(1)
                demographic_table["Standard Deviation"] = demographic_table["Standard Deviation"].round(2)
                demographic_file = f"dossier_demography_{target}_en.csv"
                demographic_table.to_csv(demographic_file, index=False, encoding="utf-8-sig")
            else:
                df_demog_filtered = pd.DataFrame()
                demographic_file = None
        else:
            df_demog_filtered = pd.DataFrame()
            demographic_file = None

        comorbidity_file = f"dossier_comorbidities_{target}_en.csv"
        comorbidities.to_csv(comorbidity_file, index=False, encoding="utf-8-sig")

        substances_table = pd.DataFrame(list(substance_use.items()), columns=["Substance", "Unique Patients (N)"])
        substances_table["Proportion (%)"] = (substances_table["Unique Patients (N)"] / total_patients * 100).round(2)
        substances_table = substances_table.sort_values("Unique Patients (N)", ascending=False)
        substances_file = f"dossier_substances_{target}_en.csv"
        substances_table.to_csv(substances_file, index=False, encoding="utf-8-sig")

        print(f"DATA TABLES EXPORTED ({specialty_name.upper()}):")
        if ciap_file:
            print(f" - '{ciap_file}'")
        if demographic_file:
            print(f" - '{demographic_file}'")
        print(f" - '{comorbidity_file}'")
        print(f" - '{substances_file}'")

        sns.set_theme(style="whitegrid")
        fig = plt.figure(figsize=(20, 14))
        highlight_color = "#2980b9" if target == "oftalmologia" else "#27ae60"

        fig.suptitle(
            f"Epidemiological Dossier: {specialty_name.upper()} Queue (N = {total_patients} unique patients)",
            fontsize=20,
            fontweight="bold",
            y=0.98,
            color=highlight_color,
        )

        ax1 = plt.subplot(2, 2, 1)
        if col_ciap:
            top_ciaps = df_target[col_ciap].dropna().value_counts().head(5)
            translated_ciaps = [translate_ciap(m) for m in top_ciaps.index]
            short_names = [str(m)[:55] + ("..." if len(str(m)) > 55 else "") for m in translated_ciaps]
            sns.barplot(x=top_ciaps.values, y=short_names, palette=f"light:{highlight_color}_r", ax=ax1)
            ax1.set_title(f"1. Main CIAP-2 Reasons for {specialty_name}", fontsize=14, fontweight="bold")
            ax1.set_xlabel("Number of Unique Patients")
            ax1.set_ylabel("")
            for p in ax1.patches:
                ax1.annotate(
                    f" {int(p.get_width())}",
                    (p.get_width(), p.get_y() + p.get_height() / 2.0),
                    va="center",
                    fontweight="bold",
                )

        ax2 = plt.subplot(2, 2, 2)
        if not df_demog_filtered.empty:
            sns.violinplot(data=df_demog_filtered, x="Sex_EN", y="Age_Num", palette="muted", inner="quart", ax=ax2)
            ax2.set_title("2. Demographic Profile: Age vs. Sex", fontsize=14, fontweight="bold")
            ax2.set_ylabel("Age (Years)")
            ax2.set_xlabel("")

        ax3 = plt.subplot(2, 2, 3)
        comorb_melt = comorbidities.melt(id_vars="Burden Category", var_name="Type", value_name="Patients")
        sns.barplot(
            data=comorb_melt,
            x="Burden Category",
            y="Patients",
            hue="Type",
            palette=["#e67e22", highlight_color],
            ax=ax3,
        )
        ax3.set_title("3. Concurrent Disease Burden (Infectious vs. Chronic)", fontsize=14, fontweight="bold")
        ax3.set_ylabel("Number of Patients")
        ax3.set_xlabel("")

        ax4 = plt.subplot(2, 2, 4)
        sr_substances = pd.Series(substance_use).sort_values(ascending=False)
        sns.barplot(x=sr_substances.values, y=sr_substances.index, palette="viridis", ax=ax4)
        ax4.set_title("4. Chemical Map in the Queue", fontsize=14, fontweight="bold")
        ax4.set_xlabel("Number of Unique Patients")
        ax4.set_ylabel("")
        for p in ax4.patches:
            ax4.annotate(
                f" {int(p.get_width())}",
                (p.get_width(), p.get_y() + p.get_height() / 2.0),
                va="center",
                fontweight="bold",
            )

        sns.despine(left=True, bottom=True)
        plt.tight_layout(pad=3.0)
        plt.show()

        print(f"\n[ REASON CHECK (CIAP-2) - {specialty_name.upper()} ]")
        if col_ciap:
            top_ciaps_console = df_target[col_ciap].dropna().value_counts().head(5)
            for ciap, qty in top_ciaps_console.items():
                pct = (qty / total_patients) * 100
                print(f"  -> {translate_ciap(ciap)[:75]:<75} | {qty} patients ({pct:.1f}%)")

        print(f"\n[ AGE CHECK - {specialty_name.upper()} ]")
        if not df_demog_filtered.empty:
            print(f"  -> Mean age calculated: {df_demog_filtered['Age_Num'].mean():.1f} years")
            print(f"  -> Youngest patient: {df_demog_filtered['Age_Num'].min():.0f} years")
            print(f"  -> Oldest patient: {df_demog_filtered['Age_Num'].max():.0f} years")
        print("=" * 100 + "\n")

# ================================================================
# 7. RUN EVERYTHING
# ================================================================
def main():
    analysis_1_top_specialty_demand(df_raw)
    analysis_2_mental_health_dossier(df_raw)
    analysis_3_target_specialty_dossiers(df_raw)

main()